Notebook to generate the ranking based on the ratio between the model and baseline WIS

In [1]:
import numpy as np 
import pandas as pd 
from aux_func import code_to_state, estado_para_regiao

In [2]:
challenge = 'chik_city'

df_preds = pd.read_csv(f'./predictions/predictions_all_models_{challenge}.csv.gz', index_col = 'Unnamed: 0')
df_preds.date = pd.to_datetime(df_preds.date)
df_preds.head()

,date,lower_95,lower_90,lower_80,lower_50,pred,upper_50,upper_80,upper_90,upper_95,adm_2,id,validation,wis,model
0,2022-10-09,0.055665,0.212458,0.433722,1.040944,2.078688,5.014152,8.274178,13.705758,19.956902,2211001,8515,1,18.427146,3rd_imdc_emap_epidematicos_prophet
1,2022-10-16,0.129924,0.150721,0.512863,0.789217,1.698967,4.608482,6.717145,8.018087,11.891967,2211001,8515,1,18.427146,3rd_imdc_emap_epidematicos_prophet
2,2022-10-23,0.167603,0.189093,0.322311,1.077777,2.078474,4.075722,6.833111,7.773644,8.318806,2211001,8515,1,18.427146,3rd_imdc_emap_epidematicos_prophet
3,2022-10-30,0.000000,0.000000,0.049948,0.593776,1.713534,3.896104,5.676596,6.242217,10.483744,2211001,8515,1,18.427146,3rd_imdc_emap_epidematicos_prophet
4,2022-11-06,0.000000,0.000000,0.000000,0.345545,1.215062,3.177605,4.522743,5.318249,8.392595,2211001,8515,1,18.427146,3rd_imdc_emap_epidematicos_prophet


In [3]:
df_preds.model.unique()

<ArrowStringArray>
[     '3rd_imdc_emap_epidematicos_prophet',
                   '3rd_imdc_nus_nus-cerm',
 '3rd_imdc_emap_epidematicos_sarimax_muni',
                 '3rd_imdc_emap_lstm_muni',
                 '3rd_imdc_emap_xgbsillas',
             '3rd_imdc_purdue_neuralearth']
Length: 6, dtype: str

In [4]:
df_preds.model.unique().shape

(6,)

In [5]:
df_agg_wis = (
    df_preds
    .groupby(["model", "adm_2", "validation"], as_index=False)["wis"]
    .mean()
)


df_agg_wis.head()

,model,adm_2,validation,wis
0,3rd_imdc_emap_epidematicos_prophet,1716109,1,50.286102
1,3rd_imdc_emap_epidematicos_prophet,1716109,2,2.681518
2,3rd_imdc_emap_epidematicos_prophet,1716109,3,1.320000
3,3rd_imdc_emap_epidematicos_prophet,1716109,4,1.330000
4,3rd_imdc_emap_epidematicos_prophet,1721000,1,7.654261


Gerando um ranking da média da diferença entre os modelos e o baseline: 

In [6]:
model_baseline = '3rd_imdc_emap_lstm_muni'

df_baseline = (
    df_agg_wis.loc[df_agg_wis["model"] == model_baseline,
           ["adm_2", "validation", "wis"]]
    .rename(columns={"wis": "wis_baseline"})
)

# Junta o WIS do baseline aos demais modelos
df_ratio = df_agg_wis.merge(
    df_baseline,
    on=["adm_2", "validation"],
    how="left"
)

# Razão WIS / Baseline
df_ratio["wis_ratio"] = (
    df_ratio["wis"] / df_ratio["wis_baseline"]
)

# Médias das razões
df_summary = (
    df_ratio
    .groupby(["adm_2", "model"], as_index=False)
    .agg(
        arithmetic_mean_ratio=("wis_ratio", "mean"),
        geometric_mean_ratio=("wis_ratio", lambda x: np.exp(np.mean(np.log(x)))),
    )
    .sort_values(["adm_2", "geometric_mean_ratio"])
)

df_summary.head()

,adm_2,model,arithmetic_mean_ratio,geometric_mean_ratio
1,1716109,3rd_imdc_emap_epidematicos_sarimax_muni,0.256352,0.127161
5,1716109,3rd_imdc_purdue_neuralearth,0.355663,0.204236
0,1716109,3rd_imdc_emap_epidematicos_prophet,0.386086,0.249331
4,1716109,3rd_imdc_nus_nus-cerm,0.445977,0.348892
3,1716109,3rd_imdc_emap_xgbsillas,0.486731,0.445957


In [7]:
#df_ratio['region'] = df_ratio['adm_1'].replace(code_to_state).replace(estado_para_regiao)

df_ratio.to_csv(f'predictions/rank_ratio_{challenge}.csv', index = False)